In [2]:
import os
from pathlib import Path

# Get the current directory of the notebook
notebook_path = Path.cwd()

ROOT = notebook_path.parent.parent

# Change the Working Directory for the whole process
os.chdir(ROOT)

print(f"Current Working Directory fixed to: {os.getcwd()}")

Current Working Directory fixed to: /srv/homes/onbo10/thesis_main


In [3]:
from utilities.visualizer import SegmentationVisualizer
import random
import json
import cv2

In [8]:
# Initialize the visualizer
viz = SegmentationVisualizer(alpha=0.4)
json_path = 'results/Segmentation/nnunet_results/nnUnet_inference/Dataset789_Endovis17_all_folds/predicted_masks/summary.json'
img_dir = 'data/EndoVis2017/Dataset789_Endovis17/imagesTs'
save_dir= 'results/Segmentation/nnunet_results/nnUnet_inference/Dataset789_Endovis17_all_folds/sample_segmentation_overlay'
os.makedirs(save_dir, exist_ok=True)
# Load the metrics
with open(json_path, 'r') as f:
    data = json.load(f)

# Your nnU-Net specific parsing
random.seed(50)
selected_entries = random.sample(data['metric_per_case'], 30)

for i, entry in enumerate(selected_entries):
    #Get the paths
    pred_path = entry['prediction_file']
    gt_path = entry['reference_file']
    base_name = os.path.basename(pred_path).replace('.png', '')
    img_path = os.path.join(img_dir, f"{base_name}_0000.png")

    #Load data
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    gt = cv2.imread(gt_path, cv2.IMREAD_GRAYSCALE)
    pred = cv2.imread(pred_path, cv2.IMREAD_GRAYSCALE)
    
    #Extract metrics
    case_metrics = {
        "Dice": entry['metrics']['1']['Dice'],
        "IoU": entry['metrics']['1']['IoU']
    }

    #Plot and/or save
    viz.plot_comparison(
        image=img, 
        gt_mask=gt, 
        pred_mask=pred, 
        title_base=base_name,
        metrics=case_metrics, save_path=os.path.join(save_dir, f"sample_{i}_{base_name}.png"),
        display=False
    )

* Inference of nnUnet on a Surgpose Sample

In [6]:
# Initialize the visualizer
viz = SegmentationVisualizer(alpha=0.4, cmap_pred='autumn')
img_dir = 'data/EndoVis2017/Surgpose_for_nnUnet_test_left_right/imagesTs'
pred_dir = 'results/Segmentation/nnunet_results/nnUnet_inference/Refined_EdgeCut_BBox'
save_dir= 'results/Segmentation/nnunet_results/nnUnet_inference/Refined_EdgeCut_BBox/sample_segmentation_overlay'
pred_files = [f for f in os.listdir(pred_dir) if f.endswith('.png')]
os.makedirs(save_dir, exist_ok=True)
print(f"Plotting {len(pred_files)} SurgPose generalization results...")

for pred_name in pred_files:
   
    base_name = pred_name.replace('.png', '')
    img_path = os.path.join(img_dir, f"{base_name}_0000.png")
    pred_path = os.path.join(pred_dir, pred_name)
    
    # Load Data
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    pred_mask = cv2.imread(pred_path, cv2.IMREAD_GRAYSCALE)
    
    viz.plot_comparison(
        image=img,
        gt_mask=None, 
        pred_mask=pred_mask,
        title_base=f"SurgPose: {base_name}",
        metrics=None, 
        save_path=os.path.join(save_dir, f"{base_name}.png"),
        display=False
    )

Plotting 180 SurgPose generalization results...
